# OCR keyframe bằng PP-OCRv6 trên Kaggle
Bật **Internet** và **GPU** trong Kaggle. Notebook tải từng ZIP keyframe từ Google Drive, clone nhánh `vbs-dev`, dùng `ExtractOCRPipeline` của repo để chạy PaddleOCR, rồi upload JSON OCR thô trực tiếp lên Drive theo lô 10 video. ZIP nguồn `<video_id>.zip` chỉ chứa ảnh `<frame_id>.jpg` ở ngay gốc; frame ID phải có đúng 6 chữ số, ví dụ `000000.jpg`. Chọn dải video ở cell đầu. Gắn `rclone.conf` vào Kaggle Input hoặc đổi đường dẫn config bên dưới.

Output mỗi video: `<video_id>.json` ở ngay gốc folder OCR. Bật `OVERWRITE_REMOTE` nếu muốn thay JSON đã có. Chạy các cell theo thứ tự.

In [ ]:
# 1. Cấu hình
import os, re, shutil, subprocess, sys, tempfile, zipfile, json, stat
from pathlib import Path
REPO_URL = 'https://github.com/iamtro-2006/event-retrieval-system.git'
REPO_BRANCH = 'vbs-dev'  # nhánh chứa engine='paddle' sau khi push
DATASET = 'V3C'  # hoặc 'LHE'
START_VIDEO_ID = 1
END_VIDEO_ID = 1
DOWNLOAD_ROOT_FOLDER_ID = '1OMEKhaXWKlZyluE-2kCVZZodh6_mHzSk'  # ZIP keyframe
UPLOAD_ROOT_FOLDER_ID = '1YgxyIh9vBKT5CbX8luRCX5egIGds67sO'    # JSON OCR thô
RCLONE_CONFIG_SOURCE = Path('/kaggle/input/datasets/vnnoname/rclone-new/rclone.conf')
RCLONE_REMOTE = 'iamtro'
OVERWRITE_REMOTE = False
UPLOAD_BATCH_SIZE = 10
BATCH_SIZE = 8  # giảm nếu GPU hết bộ nhớ

def setting(name, value):
    if value: return value
    if os.environ.get(name): return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception as exc:
        raise ValueError(f'Thiếu {name}: điền ở cell 1 hoặc Kaggle Secrets') from exc

DOWNLOAD_ROOT_FOLDER_ID = setting('DOWNLOAD_ROOT_FOLDER_ID', DOWNLOAD_ROOT_FOLDER_ID)
UPLOAD_ROOT_FOLDER_ID = setting('UPLOAD_ROOT_FOLDER_ID', UPLOAD_ROOT_FOLDER_ID)
if DOWNLOAD_ROOT_FOLDER_ID == UPLOAD_ROOT_FOLDER_ID:
    raise ValueError('Folder Drive nguồn và đích phải khác nhau')
if DATASET not in {'V3C', 'LHE'} or not 1 <= START_VIDEO_ID <= END_VIDEO_ID:
    raise ValueError('DATASET hoặc dải video ID không hợp lệ')
if not RCLONE_CONFIG_SOURCE.is_file():
    raise FileNotFoundError(f'Chưa gắn rclone config: {RCLONE_CONFIG_SOURCE}')
if not isinstance(BATCH_SIZE, int) or BATCH_SIZE < 1:
    raise ValueError('BATCH_SIZE phải là số nguyên dương')

def video_id(number):
    return f'LHE{number:02d}' if DATASET == 'LHE' else f'{number:05d}'

RUN_ROOT = Path(tempfile.mkdtemp(prefix='ocr-paddle-', dir='/kaggle/working'))
JOBS_ROOT = RUN_ROOT / 'jobs'
JOBS_ROOT.mkdir()
STAGE_DIR = RUN_ROOT / 'pending_json'
STAGE_DIR.mkdir()
RCLONE_CONFIG = RUN_ROOT / 'rclone.conf'
shutil.copyfile(RCLONE_CONFIG_SOURCE, RCLONE_CONFIG)
RCLONE_CONFIG.chmod(0o600)
REPO_DIR = RUN_ROOT / 'event-retrieval-system'
BACKEND_DIR = REPO_DIR / 'backend'
print('Workspace:', RUN_ROOT)


In [ ]:
# 2. Clone code, cài OCR trong venv riêng và chuẩn bị rclone
def run(args, *, cwd=None, env=None):
    args = [str(x) for x in args]
    print('+', ' '.join(args[:4]), '...' if len(args) > 4 else '', flush=True)
    return subprocess.run(args, cwd=cwd, env=env, check=True, text=True)

if not REPO_DIR.exists():
    run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR])
branch = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
if branch != REPO_BRANCH:
    raise RuntimeError(f'Checkout đang ở nhánh {branch}, cần {REPO_BRANCH}')
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
ocr_config = BACKEND_DIR / 'configs' / 'ocr_extraction.yaml'
if not ocr_config.is_file() or 'self.engine == "paddle"' not in (BACKEND_DIR / 'src/ocr_extraction/pipeline/extract_ocr.py').read_text(encoding='utf-8'):
    raise RuntimeError('Nhánh GitHub chưa có OCR engine paddle; hãy push code rồi chạy lại')
print('Commit:', commit)

# Paddle GPU 3.2.0 và Torch GPU của Kaggle yêu cầu NCCL khác nhau. Venv dùng Torch CPU
# cho DataLoader/modelscope; chỉ Paddle chạy GPU. Không thay CUDA/Torch của kernel Kaggle.
run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'])
run([sys.executable, '-m', 'uv', 'python', 'install', '3.12'])
OCR_VENV = RUN_ROOT / 'venv-ocr'
OCR_PYTHON = OCR_VENV / 'bin' / 'python'
if not OCR_PYTHON.is_file():
    run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', '--seed', OCR_VENV])
run([OCR_PYTHON, '-m', 'pip', 'install', '--quiet', 'paddlepaddle-gpu==3.2.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'])
run([OCR_PYTHON, '-m', 'pip', 'install', '--quiet', 'torch', '--index-url', 'https://download.pytorch.org/whl/cpu'])
# PaddleX tự ràng buộc PyYAML/OpenCV; để resolver chọn đúng phiên bản.
run([OCR_PYTHON, '-m', 'pip', 'install', 'paddleocr==3.7.0'])
run([OCR_PYTHON, '-m', 'pip', 'install', 'tqdm', 'Pillow'])
run([OCR_PYTHON, '-c', 'import paddle, paddleocr, torch, cv2, yaml; assert paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0; print("Paddle", paddle.__version__, "Torch CPU", torch.__version__, "PaddleOCR", paddleocr.__version__)'])

rclone = shutil.which('rclone')
if rclone is None and shutil.which('apt-get'):
    try:
        run(['apt-get', 'update', '-qq'])
        run(['apt-get', 'install', '-y', '-qq', 'rclone'])
        rclone = shutil.which('rclone')
    except subprocess.CalledProcessError:
        pass
if rclone is None:
    import urllib.request
    release = 'v1.75.2'
    archive = RUN_ROOT / f'rclone-{release}-linux-amd64.zip'
    url = f'https://github.com/rclone/rclone/releases/download/{release}/{archive.name}'
    with urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'kaggle-ocr'}), timeout=120) as src, archive.open('wb') as dst:
        shutil.copyfileobj(src, dst)
    with zipfile.ZipFile(archive) as zf:
        matches = [item for item in zf.infolist() if item.filename.endswith('/rclone') and not item.is_dir()]
        if len(matches) != 1:
            raise RuntimeError('Không tìm thấy binary rclone trong ZIP release')
        rclone = RUN_ROOT / 'rclone'
        with zf.open(matches[0]) as src, rclone.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    rclone.chmod(0o755)
if rclone is None:
    raise RuntimeError('Cài rclone không thành công')
run([rclone, 'version'])


In [ ]:
# 3. Liệt kê ZIP nguồn và chọn video ID
def remote_args(folder_id):
    return [rclone, '--config', RCLONE_CONFIG, '--drive-root-folder-id', folder_id]

source_listing = subprocess.check_output(
    [*map(str, remote_args(DOWNLOAD_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'], text=True)
source_names = set(source_listing.splitlines())
selected = [video_id(i) for i in range(START_VIDEO_ID, END_VIDEO_ID + 1)]
missing = [vid for vid in selected if f'{vid}.zip' not in source_names]
if missing:
    raise FileNotFoundError(f'Thiếu ZIP keyframe trong Drive nguồn: {missing}')
dest_listing = subprocess.check_output(
    [*map(str, remote_args(UPLOAD_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'], text=True)
existing = set(dest_listing.splitlines())
pending = [vid for vid in selected if OVERWRITE_REMOTE or f'{vid}.json' not in existing]
print('Sẽ xử lý:', pending, '| bỏ qua output đã có:', len(selected) - len(pending))


In [ ]:
# 4. Xử lý tối đa 10 video mỗi lượt OCR, rồi upload JSON của cả lượt
import yaml
FRAME_PATTERN = re.compile(r'^[0-9]{6}\.jpg$')

def prepare_video(vid, batch_work):
    source_zip = batch_work / f'{vid}.zip'
    run([*remote_args(DOWNLOAD_ROOT_FOLDER_ID), 'copyto', f'{RCLONE_REMOTE}:{vid}.zip', source_zip])
    if not source_zip.is_file() or source_zip.stat().st_size == 0:
        raise RuntimeError(f'ZIP nguồn rỗng: {vid}')
    frame_dir = batch_work / 'keyframes' / DATASET / vid
    frame_dir.mkdir(parents=True)
    with zipfile.ZipFile(source_zip) as zf:
        if zf.testzip() is not None:
            raise ValueError(f'ZIP nguồn lỗi CRC: {vid}')
        images = [info for info in zf.infolist() if not info.is_dir()]
        names = [Path(info.filename).stem for info in images]
        if not images or len(names) != len(set(names)):
            raise ValueError(f'{vid}: ZIP không có ảnh hoặc tên frame trùng')
        for info in images:
            path = Path(info.filename)
            if (len(path.parts) != 1 or '\\' in info.filename or
                    not FRAME_PATTERN.fullmatch(info.filename) or stat.S_ISLNK(info.external_attr >> 16)):
                raise ValueError(f'{vid}: ZIP chỉ được chứa ảnh 6 chữ số .jpg ở ngay gốc: {info.filename}')
            with zf.open(info) as src, (frame_dir / path.name).open('wb') as dst:
                shutil.copyfileobj(src, dst)
    source_zip.unlink()  # ảnh đã giải nén, tiết kiệm dung lượng Kaggle
    return set(names)

def validate_result(vid, expected, batch_work):
    result = batch_work / 'ocr_output' / DATASET / f'{vid}.json'
    if not result.is_file():
        raise FileNotFoundError(f'Không có OCR JSON: {result}')
    data = json.loads(result.read_text(encoding='utf-8'))
    if set(data) != expected or any(not isinstance(entry, list) or len(entry) != 2 or
        len(entry[0]) != len(entry[1]) for entry in data.values()):
        raise RuntimeError(f'{vid}: OCR JSON thiếu frame hoặc boxes/texts không khớp')
    shutil.move(result, STAGE_DIR / f'{vid}.json')
    return len(data)

def upload_batch():
    staged = sorted(STAGE_DIR.glob('*.json'))
    if not staged:
        return
    run([*remote_args(UPLOAD_ROOT_FOLDER_ID), 'copy', STAGE_DIR, f'{RCLONE_REMOTE}:',
         '--transfers', str(min(UPLOAD_BATCH_SIZE, len(staged))), '--checkers', '10'])
    run([*remote_args(UPLOAD_ROOT_FOLDER_ID), 'check', STAGE_DIR, f'{RCLONE_REMOTE}:', '--one-way'])
    for path in staged:
        path.unlink()
    print(f'Uploaded {len(staged)} JSON thô')

completed = {}
for start in range(0, len(pending), UPLOAD_BATCH_SIZE):
    batch_ids = pending[start:start + UPLOAD_BATCH_SIZE]
    batch_work = JOBS_ROOT / f'batch_{start // UPLOAD_BATCH_SIZE:04d}'
    if batch_work.exists() or list(STAGE_DIR.glob('*.json')):
        raise RuntimeError(f'Có dữ liệu chưa upload từ lần chạy trước: {batch_work} hoặc {STAGE_DIR}')
    batch_work.mkdir()
    expected = {vid: prepare_video(vid, batch_work) for vid in batch_ids}
    cfg = yaml.safe_load(ocr_config.read_text(encoding='utf-8'))
    cfg['extraction'].update(engine='paddle', recognizer='paddleocr', batch_size=BATCH_SIZE,
                             input_keyframes_root=str(batch_work / 'keyframes'),
                             output_root=str(batch_work / 'ocr_output'),
                             skip_existing=False, visualized=False)
    cfg['extraction']['model']['device'] = 'gpu:0'  # tên model lấy từ configs/ocr_extraction.yaml
    runtime_cfg = batch_work / 'ocr.runtime.yaml'
    runtime_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    worker_env = {**os.environ, 'PYTHONPATH': str(BACKEND_DIR)}
    run([OCR_PYTHON, '-m', 'scripts.ocr.extract.run', '--config', runtime_cfg],
        cwd=BACKEND_DIR, env=worker_env)
    for vid in batch_ids:
        completed[vid] = validate_result(vid, expected[vid], batch_work)
        print(f'OCR xong {vid}: {completed[vid]} keyframes; chờ upload')
    upload_batch()
    shutil.rmtree(batch_work)  # chỉ dọn sau khi batch upload và check thành công
print('Xong:', completed, '| commit:', commit)
